In [ ]:
### Load common inputs and create the figure-data output directories
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import function
import os

for i in range(6):
    os.makedirs(f"../results/book/{i+1}", exist_ok=True)
    os.makedirs(f"../results/book/s{i+1}", exist_ok=True)

yield_data = pd.read_csv("../data/yield.csv", engine="python").values[:, 1:]
social_data = pd.read_csv("../data/social.csv", engine="python").values[:, 1:]
natural_dataset = pd.read_csv("../data/natural.csv", engine="python")
natural_data = function.natural_feature_builder(natural_dataset).values

yield_data = torch.from_numpy(yield_data.astype("float")).float()
social_data = torch.from_numpy(social_data.astype("float")).float().reshape(-1, 22, 9)
natural_data = torch.from_numpy(natural_data.astype("float")).float().reshape(22, 12, -1, 8)[:, :9].permute(2, 0, 1, 3).reshape(654, 22, -1)

county_index, counts = np.arange(654).reshape(-1, 1), [117, 76, 64, 57, 89, 109, 67, 75]
county_index = np.concatenate([county_index, np.repeat(range(len(counts)), counts).reshape(-1, 1)], 1)
yield_data, social_data, natural_data, county_index = function.filter_(yield_data, social_data, natural_data, county_index)
yield_obsvd, yield_trend, yield_resid = function.resolve(yield_data)
yield_data, social_data, natural_data, county_index = function.flatten(yield_data, social_data, natural_data, county_index)

province_iter = ["hebei", "shanxi", "jiangsu", "anhui", "shandong", "henan", "hubei", "shaanxi"]
yield_province = pd.DataFrame(np.concatenate([county_index[:, :2], yield_data.numpy()], 1), columns=["county", "province", "yield"])
yield_province = yield_province.groupby(["county", "province"], as_index=False)["yield"].mean()
county_bin_province = {}
for i, p in enumerate(province_iter):
    yield_p = yield_province[yield_province["province"] == i][["county", "yield"]].copy()
    yield_p["bin"] = pd.cut(yield_p["yield"], bins=3, labels=False)
    county_bin_province[p] = yield_p[["county", "bin"]]

county_bin_period = {}
for i in range(2):
    p, index = ["early", "later"][i], [np.arange(11), np.arange(11, 22)][i]
    mask = np.isin(county_index[:, 2], index)
    exec(f"yield_data_{p}, county_index_{p} = yield_data[mask], county_index[mask]")
    yield_period = pd.DataFrame(np.concatenate([county_index[mask, 0].reshape(-1, 1), yield_data[mask].numpy()], 1), columns=["county", "yield"])
    yield_period = yield_period.groupby("county", as_index=False)["yield"].mean()
    yield_period["bin"] = pd.cut(yield_period["yield"], bins=3, labels=False)
    county_bin_period[p] = yield_period[["county", "bin"]]

In [ ]:
### Prepare yield-stratification and group-wise prediction-error datasets
yield_csv = pd.DataFrame(np.concatenate([county_index[:, 0].reshape(-1, 1), yield_data.numpy()], 1), columns=["county", "yield"])
yield_grouped = yield_csv.groupby("county")["yield"].agg(["mean", "std"])
yield_grouped["cv"] = yield_grouped["std"] / yield_grouped["mean"]
yield_grouped.reset_index(inplace=True)
yield_mapping = yield_grouped.sort_values(by="county")[["county", "mean"]]
yield_mapping.to_csv("../results/book/1/yield_mapping.csv", index=False)

yield_grouped["bin"] = pd.cut(yield_grouped["mean"], bins=3, labels=False)
county_bin = yield_grouped[["county", "bin"]].copy()
yield_edges = pd.cut(yield_grouped["mean"], bins=3, labels=False, retbins=True)[1]
yield_t1, yield_t2 = yield_edges[1], yield_edges[2]
yield_t1 = (yield_grouped["mean"][yield_grouped["bin"] == 0].max() + yield_grouped["mean"][yield_grouped["bin"] == 1].min()) / 2
yield_t2 = (yield_grouped["mean"][yield_grouped["bin"] == 1].max() + yield_grouped["mean"][yield_grouped["bin"] == 2].min()) / 2
print(f"yield_t1, yield_t2 = {yield_t1}, {yield_t2}")
yield_grouped = yield_grouped.sort_values(["bin", "county"]).reset_index(drop=True)[["county", "bin", "mean", "std", "cv"]]
yield_grouped.to_csv("../results/book/1/yield_grouped.csv", index=False)

model_iter, error_grouped = ["dnn", "cnn", "rnn", "lst", "gru", "att"], None
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_inst_{model}.csv", engine="python").groupby("county", as_index=False).mean()
    if error_grouped is None:
        error_grouped = error_csv[["county"]].merge(county_bin, on="county", how="left", validate="one_to_one")
    mse_base_pret = error_csv[["county", "mse_base_pret"]].rename(columns={"mse_base_pret": f"mse_{model}"})
    error_grouped = error_grouped.merge(mse_base_pret, on="county", how="left")
error_grouped = error_grouped.sort_values(["bin", "county"]).reset_index(drop=True)
error_grouped.to_csv("../results/book/1/error_grouped.csv", index=False)

error_dataset = []
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_accu_{model}.csv", engine="python")
    error_base_pret = error_csv.loc[error_csv["Unnamed: 0"] == "pret", ["r2_base", "mse_base", "mae_base"]].copy()
    error_base_pret.columns = ["r2", "mse", "mae"]
    error_base_pret.insert(0, "model", model)
    error_dataset.append(error_base_pret)
error_dataset = pd.concat(error_dataset, ignore_index=True)
error_dataset.to_csv("../results/book/1/error_dataset.csv", index=False)

error_dataset, error_grouped = [], []
for model in model_iter:
    file_list1 = [(f"kf{i}", f"../results/work/1/test_accu_{model}_kf{i}.csv") for i in range(1, 6)]
    file_list1 += [("mean", f"../results/work/1/test_accu_{model}.csv")]
    for index, file in file_list1:
        error_csv = pd.read_csv(file, engine="python")
        error_base_pret = error_csv.loc[error_csv["Unnamed: 0"] == "pret", ["r2_base", "mse_base", "mae_base"]].copy()
        error_base_pret.columns = ["r2", "mse", "mae"]
        error_base_pret.insert(0, "index", index)
        error_base_pret.insert(0, "model", model)
        error_dataset.append(error_base_pret)
    file_list2 = [(f"kf{i}", f"../results/work/1/test_inst_{model}_kf{i}.csv") for i in range(1, 6)]
    file_list2 += [("mean", f"../results/work/1/test_inst_{model}.csv")]
    for index, file in file_list2:
        error_csv = pd.read_csv(file, engine="python").groupby("county", as_index=False).mean()
        error_csv = error_csv.merge(county_bin, on="county", how="left", validate="one_to_one")
        mse_base_pret = error_csv.groupby("bin", observed=False)["mse_base_pret"].mean().reindex([0, 1, 2])
        error_grouped.append({"model": model, "index": index, "mse_low": mse_base_pret.loc[0], "mse_moderate": mse_base_pret.loc[1], "mse_high": mse_base_pret.loc[2]})
error_dataset, error_grouped = pd.concat(error_dataset, ignore_index=True), pd.DataFrame(error_grouped)
error_summary = error_dataset.merge(error_grouped, on=["model", "index"], how="inner")
error_summary["mse_low_diff"] = error_summary["mse_low"] - error_summary["mse"]
error_summary["mse_moderate_diff"] = error_summary["mse_moderate"] - error_summary["mse"]
error_summary["mse_high_diff"] = error_summary["mse_high"] - error_summary["mse"]
error_summary = error_summary[["model", "index", "r2", "mse", "mae", "mse_low", "mse_low_diff", "mse_moderate", "mse_moderate_diff", "mse_high", "mse_high_diff"]]
error_summary.to_csv("../results/book/s1/error_summary.csv", index=False)

social_columns, natural_columns = ["PopD", "HeaR", "PuPr", "TePr", "FiRv", "FiEx", "ReSv", "InLn"], ["Radn", "AirT", "SoTe", "AirH", "SoMo", "Prec", "SuPr", "WiSp"]
social_csv = pd.DataFrame(np.concatenate([county_index[:, [0, 2]], social_data.numpy()], 1), columns=["county", "year"] + social_columns)
natural_csv = pd.DataFrame(np.concatenate([county_index[:, [0, 2]], natural_data.reshape(-1, 9, 8).mean(1).numpy()], 1), columns=["county", "year"] + natural_columns)
feature_csv = pd.concat([natural_csv, social_csv.iloc[:, 2:]], axis=1)
feat_mapping = feature_csv.groupby("county").agg("mean").reset_index()[["county"] + natural_columns + social_columns]
feat_mapping.to_csv("../results/book/s1/feat_mapping.csv", index=False)

feat_grouped = feature_csv.iloc[:, 1:].groupby("year").agg("mean").reset_index()
feat_grouped.to_csv("../results/book/s1/feat_grouped.csv", index=False)

yield_t1, yield_t2 = 3238.2259283768503, 5548.382456568668


In [ ]:
### Prepare yield-component and feature datasets for association analyses
yield_columns = ["YO", "TY", "DY"]
social_columns, natural_columns = ["PopD", "HeaR", "PuPr", "TePr", "FiRv", "FiEx", "ReSv", "InLn"], ["Radn", "AirT", "SoTe", "AirH", "SoMo", "Prec", "SuPr", "WiSp"]
yield_csv = pd.DataFrame(np.concatenate([county_index[:, 2].reshape(-1, 1), yield_data.numpy()], 1), columns=["year", "yield"])
yield_yo = torch.from_numpy(yield_csv.groupby("year").mean().to_numpy().copy()).permute(1, 0)
yield_yo, yield_ty, yield_dy = function.resolve(yield_yo)
yield_grouped = pd.DataFrame(torch.concatenate([yield_yo, yield_ty, yield_dy], 1), columns=yield_columns)
yield_grouped.to_csv("../results/book/2/yield_grouped.csv", index=False)

yield_csv = np.concatenate([yield_obsvd.numpy(), yield_trend.numpy(), yield_resid.numpy()], 1)
yield_csv = pd.DataFrame(np.concatenate([county_index[:, [0, 2]], yield_csv], 1), columns=["county", "year"] + yield_columns)
social_csv = pd.DataFrame(np.concatenate([county_index[:, [0, 2]], social_data.numpy()], 1), columns=["county", "year"] + social_columns)
natural_csv = pd.DataFrame(np.concatenate([county_index[:, [0, 2]], natural_data.reshape(-1, 9, 8).mean(1).numpy()], 1), columns=["county", "year"] + natural_columns)
for i in ["yield", "social", "natural"]:
    exec(f"dataset, columns = pd.DataFrame([]), {i}_columns")
    for j in columns:
        exec(f"data_scaled = {i}_csv.pivot(index = 'year', columns = 'county', values = '{j}')")
        data_scaled = StandardScaler().fit_transform(data_scaled.values)
        data_scaled = torch.from_numpy(data_scaled.reshape(-1)).float()
        index_notnan = np.arange(data_scaled.size(0))[~torch.isnan(data_scaled)]
        data_scaled = data_scaled[index_notnan]
        exec(f"dataset['{j}'] = data_scaled")
    exec(f"{i}_dataset = dataset")
yield_dataset.to_csv("../results/book/2/yield_dataset.csv", index=False)
feat_dataset = pd.concat([social_dataset, natural_dataset], axis=1)
feat_dataset.to_csv("../results/book/2/feat_dataset.csv", index=False)

feature_book, feat_imptnce = None, None
for model in model_iter:
    feat = []
    for i in ["obsvd", "resid"]:
        feat_mean = pd.read_csv(f"../results/work/2/{i}_{model}.csv", engine="python").abs().values[:, 2:].mean(0)
        feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9, 8).mean(0)])
        feat_mean = feat_mean / feat_mean.sum()
        feat.append(feat_mean)
    feat_csv = pd.DataFrame(np.array(feat).T, index=social_columns + natural_columns, columns=["obsvd", "resid"])
    lsfg_csv = pd.DataFrame(feat_csv.values[:8].sum(0).reshape(1, 2), index=["LSFG"], columns=["obsvd", "resid"])
    nefg_csv = pd.DataFrame(feat_csv.values[8:].sum(0).reshape(1, 2), index=["NEFG"], columns=["obsvd", "resid"])
    book_model = pd.concat([feat_csv.iloc[:8], lsfg_csv, feat_csv.iloc[8:], nefg_csv]).rename_axis("feat").reset_index()
    ipts_model = feat_csv.reindex(natural_columns + social_columns[::-1]).rename_axis("feat").reset_index()
    book_model = book_model.rename(columns={"obsvd": f"{model}_yo", "resid": f"{model}_dy"})
    ipts_model = ipts_model.rename(columns={"obsvd": f"{model}_yo", "resid": f"{model}_dy"})
    feature_book = book_model if feature_book is None else feature_book.merge(book_model, on="feat", how="left")
    feat_imptnce = ipts_model if feat_imptnce is None else feat_imptnce.merge(ipts_model, on="feat", how="left")
feature_book.to_csv("../results/book/2/feature_book.csv", index=False)
feat_imptnce.to_csv("../results/book/2/feat_imptnce.csv", index=False)

county_index_csv = pd.DataFrame(county_index, columns=["county", "province", "year"])
province_index = county_index_csv.pivot(index="year", columns="county", values="province").values
province_index = torch.from_numpy(province_index.reshape(-1).copy()).float()
index_notnan = np.arange(province_index.size(0))[~torch.isnan(province_index)]
province_index = province_index[index_notnan].int().numpy()
for i, p in enumerate(province_iter):
    yield_dataset_p, feat_dataset_p = yield_dataset[province_index == i], feat_dataset[province_index == i]
    yield_dataset_p.to_csv(f"../results/book/s2/{p}_yield_dataset.csv", index=False)
    feat_dataset_p.to_csv(f"../results/book/s2/{p}_feat_dataset.csv", index=False)
    feat_imptnce = None
    for model in model_iter:
        feat = []
        for j in ["obsvd", "resid"]:
            feat_mean = pd.read_csv(f"../results/work/4/{p}_{j}_{model}.csv", engine="python").abs().values[:, 2:].mean(0)
            feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9, 8).mean(0)])
            feat_mean = feat_mean / feat_mean.sum()
            feat.append(feat_mean)
        feat_csv = pd.DataFrame(np.array(feat).T, index=social_columns + natural_columns, columns=["obsvd", "resid"])
        feat_model = feat_csv.reindex(natural_columns + social_columns[::-1]).rename_axis("feat").reset_index()
        feat_model = feat_model.rename(columns={"obsvd": f"{model}_yo", "resid": f"{model}_dy"})
        if feat_imptnce is None:
            feat_imptnce = feat_model
        else:
            feat_imptnce = feat_imptnce.merge(feat_model, on="feat", how="left")
    feat_imptnce.to_csv(f"../results/book/s2/{p}_feat_imptnce.csv", index=False)

In [ ]:
### Prepare yield-distribution and sample-level FGAA-score datasets
error_csv = pd.read_csv("../results/work/1/test_inst_dnn.csv", engine="python")
error_csv["bin"] = pd.cut(error_csv["yield"], bins=3, labels=False)
error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
count = error_grouped.xs("count", level=1, axis=1)[["yield"]]
error = error_grouped.xs("mean", level=1, axis=1)[["mse_base_pret"]]
error_grouped = pd.concat([error_grouped[["bin"]], count, error], axis=1)
error_grouped.columns = ["bin", "yield", "mse"]
error_grouped.to_csv("../results/book/3/error_grouped1.csv", index=False)

yield_csv = pd.DataFrame(np.concatenate([county_index[:, 1].reshape(-1, 1), yield_data.numpy()], 1), columns=["province", "yield"])
yield_csv.to_csv("../results/book/3/yield_dataset.csv", index=False)

bins = range(0, 9001, 500)
bin_centers, error_grouped = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)], None
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_inst_{model}.csv", engine="python")
    error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
    error_model = error_csv.groupby("bin", observed=False).agg(count=("yield", "count"), base=("mse_base_pret", "mean"), addi=("mse_addi_pret", "mean")).reset_index()
    error_model = error_model.rename(columns={"base": f"{model}_base", "addi": f"{model}_addi"})
    if error_grouped is None:
        error_grouped = error_model
    else:
        error_grouped = error_grouped.merge(error_model.drop(columns="count"), on="bin", how="left")
error_grouped.to_csv("../results/book/3/error_grouped2.csv", index=False)

error_dataset = []
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_accu_{model}.csv", engine="python")
    error_csv.columns = ["index", "r2_base", "mse_base", "mae_base", "r2_addi", "mse_addi", "mae_addi"]
    error_pret = error_csv.loc[error_csv["index"] == "pret", ["r2_base", "mse_base", "mae_base", "r2_addi", "mse_addi", "mae_addi"]].copy()
    error_pret.insert(0, "model", model)
    error_dataset.append(error_pret)
error_dataset = pd.concat(error_dataset, ignore_index=True)
error_dataset.to_csv("../results/book/3/error_dataset.csv", index=False)

for model in model_iter:
    error_grouped = None
    file_list = [(f"kf{i}", f"../results/work/1/test_inst_{model}_kf{i}.csv") for i in range(1, 6)]
    file_list += [("mean", f"../results/work/1/test_inst_{model}.csv")]
    for index, file in file_list:
        error_csv = pd.read_csv(file, engine="python")
        error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
        error_model = error_csv.groupby("bin", observed=False).agg(count=("yield", "count"), base=("mse_base_pret", "mean"), addi=("mse_addi_pret", "mean")).reset_index()
        error_model = error_model.rename(columns={"base": f"{index}_base", "addi": f"{index}_addi"})
        if error_grouped is None:
            error_grouped = error_model
        else:
            error_grouped = error_grouped.merge(error_model.drop(columns="count"), on="bin", how="left")
    error_grouped.to_csv(f"../results/book/s3/error_grouped_{model}.csv", index=False)

In [ ]:
### Prepare provincial yield, feature, and feature-contribution datasets
yield_csv = pd.DataFrame(np.concatenate([county_index[:, 1].reshape(-1, 1), yield_data.numpy()], 1), columns=["province", "yield"])
yield_csv.to_csv("../results/book/4/yield_dataset.csv", index=False)

bins, error_grouped_gather = range(0, 9001, 500), []
bin_centers = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)]
for i in range(5):
    error_csv = pd.read_csv(f"../results/work/1/train_inst_dnn_kf{i+1}.csv", engine="python")
    error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
    error_grouped_gather.append([error_grouped.values])
error_grouped_gather = np.concatenate(error_grouped_gather, 0)
error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped.columns)
count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
error = error_grouped_gather.xs("mean", level=1, axis=1)[["mse_base_init", "mse_base_pret", "scores_base_pret", "mse_base_fine"]]
error_grouped_gather = pd.concat([error_grouped_gather[["bin"]], count, error], axis=1)
error_grouped_gather.columns = ["bin", "train_count", "train_mse_init", "train_mse_pret", "train_scores", "train_mse_fine"]

error_csv = pd.read_csv("../results/work/1/test_inst_dnn.csv", engine="python")
error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
count = error_grouped.xs("count", level=1, axis=1)[["yield"]]
error = error_grouped.xs("mean", level=1, axis=1)[["mse_base_init", "mse_base_pret", "mse_base_fine"]]
error_grouped = pd.concat([error_grouped[["bin"]], count, error], axis=1)
error_grouped.columns = ["bin", "test_count", "test_mse_init", "test_mse_pret", "test_mse_fine"]
error_grouped1 = error_grouped_gather.merge(error_grouped, on="bin", how="outer").sort_values("bin")
error_grouped1.to_csv("../results/book/4/error_grouped1.csv", index=False)

error_dataset = None
for i in ["train", "test"]:
    error_csv = pd.read_csv(f"../results/work/1/{i}_accu_dnn.csv", engine="python").iloc[:, :4]
    error_csv.columns = ["index", f"{i}_r2", f"{i}_mse", f"{i}_mae"]
    if error_dataset is None:
        error_dataset = error_csv
    else:
        error_dataset = error_dataset.merge(error_csv, on="index", how="inner")
error_dataset.to_csv("../results/book/4/error_dataset.csv", index=False)

error_grouped_gather = []
for i in range(5):
    error_csv = pd.read_csv(f"../results/work/1/train_inst_dnn_kf{i+1}.csv", engine="python")
    error_csv["bin"] = pd.cut(error_csv["yield"], bins=3, labels=False)
    error_grouped = error_csv.groupby("bin", observed=False).agg("mean").reset_index()
    error_grouped_gather.append([error_grouped.values])
error_grouped_gather = np.concatenate(error_grouped_gather, 0)
error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped.columns)
error_grouped_gather = error_grouped_gather[["mse_base_init", "mse_base_pret", "mse_base_fine"]].T
error_grouped_gather.columns, error_grouped_gather.index = ["train_low", "train_medium", "train_high"], ["mse_init", "mse_pret", "mse_fine"]

error_csv = pd.read_csv("../results/work/1/test_inst_dnn.csv", engine="python")
error_csv["bin"] = pd.cut(error_csv["yield"], bins=3, labels=False)
error_grouped = error_csv.groupby("bin", observed=False).agg("mean").reset_index()
error_grouped = error_grouped[["mse_base_init", "mse_base_pret", "mse_base_fine"]].T
error_grouped.columns, error_grouped.index = ["test_low", "test_medium", "test_high"], ["mse_init", "mse_pret", "mse_fine"]
error_grouped2 = error_grouped_gather.join(error_grouped)
error_grouped2.index.name = "index"
error_grouped2.to_csv("../results/book/4/error_grouped2.csv")

score_grouped = None
for p in province_iter:
    bins, error_grouped_gather = range(0, 9001, 500), []
    bin_centers = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)]
    for i in range(5):
        error_csv = pd.read_csv(f"../results/work/3/{p}_train_inst_dnn_kf{i+1}.csv", engine="python")
        error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
        error_grouped_kf = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
        error_grouped_gather.append([error_grouped_kf.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, axis=0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped_kf.columns)
    count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
    score = error_grouped_gather.xs("mean", level=1, axis=1)[["scores_base_pret", "scores_addi_pret"]]
    score_grouped_province = pd.concat([error_grouped_gather[["bin"]], count, score], axis=1)
    score_grouped_province.columns = ["bin", f"{p}_count", f"{p}_scores_base", f"{p}_scores_addi"]
    if score_grouped is None:
        score_grouped = score_grouped_province
    else:
        score_grouped = score_grouped.merge(score_grouped_province, on="bin", how="left")
score_grouped.to_csv("../results/book/4/score_grouped.csv", index=False)

error_dataset = []
for model in model_iter[1:]:
    bins, error_grouped_gather = range(0, 9001, 500), []
    bin_centers = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)]
    for i in range(5):
        error_csv = pd.read_csv(f"../results/work/1/train_inst_{model}_kf{i+1}.csv", engine="python")
        error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
        error_grouped_gather.append([error_grouped.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, 0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped.columns)
    count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
    error = error_grouped_gather.xs("mean", level=1, axis=1)[["mse_base_init", "mse_base_pret", "mse_base_fine"]]
    error_grouped_gather = pd.concat([error_grouped_gather[["bin"]], count, error], axis=1)
    error_grouped_gather.columns = ["bin", "train_count", "train_mse_init", "train_mse_pret", "train_mse_fine"]

    error_csv = pd.read_csv(f"../results/work/1/test_inst_{model}.csv", engine="python")
    error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
    count = error_grouped.xs("count", level=1, axis=1)[["yield"]]
    error = error_grouped.xs("mean", level=1, axis=1)[["mse_base_init", "mse_base_pret", "mse_base_fine"]]
    error_grouped = pd.concat([error_grouped[["bin"]], count, error], axis=1)
    error_grouped.columns = ["bin", "test_count", "test_mse_init", "test_mse_pret", "test_mse_fine"]
    error_grouped = error_grouped_gather.merge(error_grouped, on="bin", how="outer").sort_values("bin")
    error_grouped.to_csv(f"../results/book/s4/{model}_error_grouped.csv", index=False)

    error_model = None
    for i in ["train", "test"]:
        error_csv = pd.read_csv(f"../results/work/1/{i}_accu_{model}.csv", engine="python").iloc[:, :4]
        error_csv.columns = ["index", f"{i}_r2", f"{i}_mse", f"{i}_mae"]
        if error_model is None:
            error_model = error_csv
        else:
            error_model = error_model.merge(error_csv, on="index", how="inner")
    error_model.insert(0, "model", model)
    error_dataset.append(error_model)

    score_grouped = None
    for p in province_iter:
        bins, error_grouped_gather = range(0, 9001, 500), []
        bin_centers = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)]
        for i in range(5):
            error_csv = pd.read_csv(f"../results/work/3/{p}_train_inst_{model}_kf{i+1}.csv", engine="python")
            error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
            error_grouped_kf = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
            error_grouped_gather.append([error_grouped_kf.values])
        error_grouped_gather = np.concatenate(error_grouped_gather, 0)
        error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
        error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped_kf.columns)
        count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
        score = error_grouped_gather.xs("mean", level=1, axis=1)[["scores_base_pret", "scores_addi_pret"]]
        score_grouped_gather = pd.concat([error_grouped_gather[["bin"]], count, score], axis=1)
        score_grouped_gather.columns = ["bin", f"{p}_count", f"{p}_scores_base", f"{p}_scores_addi"]
        if score_grouped is None:
            score_grouped = score_grouped_gather
        else:
            score_grouped = score_grouped.merge(score_grouped_gather, on="bin", how="left")
    score_grouped.to_csv(f"../results/book/s4/{model}_score_grouped.csv", index=False)
error_dataset = pd.concat(error_dataset, ignore_index=True)
error_dataset.to_csv("../results/book/s4/error_dataset.csv", index=False)

In [ ]:
### Aggregate proposed-framework performance and county-level spatial-error datasets
error_grouped = None
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_inst_{model}.csv", engine="python").groupby("county", as_index=False).mean()
    if error_grouped is None:
        error_grouped = error_csv[["county"]].merge(county_bin, on="county", how="left", validate="one_to_one")
    error_model = error_csv[["county", "mse_base_pret", "mse_addi_fine"]].rename(columns={"mse_base_pret": f"{model}_base", "mse_addi_fine": f"{model}_prop"})
    error_grouped = error_grouped.merge(error_model, on="county", how="left")
error_grouped = error_grouped.sort_values(["bin", "county"]).reset_index(drop=True)
error_grouped.to_csv("../results/book/5/error_grouped.csv", index=False)

error_dataset = []
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_accu_{model}.csv", engine="python")
    base, prop = error_csv.loc[error_csv["Unnamed: 0"] == "pret"].iloc[0], error_csv.loc[error_csv["Unnamed: 0"] == "fine"].iloc[0]
    error_dataset.append(
        {
            "model": model,
            "r2_base": base["r2_base"],
            "mse_base": base["mse_base"],
            "mae_base": base["mae_base"],
            "r2_prop": prop["r2_addi"],
            "mse_prop": prop["mse_addi"],
            "mae_prop": prop["mae_addi"],
        }
    )
error_dataset = pd.DataFrame(error_dataset)
error_dataset.to_csv("../results/book/5/error_dataset.csv", index=False)

county_list = pd.DataFrame({"county": sorted(set(county_index[:, 0]))})
error_mapping = county_list.copy()
for model in model_iter:
    error_csv = pd.read_csv(f"../results/work/1/test_inst_{model}.csv", engine="python")
    error_model = error_csv.groupby("county", as_index=False).mean()[["county", "mse_base_pret", "mse_addi_fine"]]
    error_model = county_list.merge(error_model, on="county", how="left")
    error_model.columns = ["county", "mse_base", "mse_prop"]
    for i in ["base", "prop"]:
        error_model[f"{model}_{i}"] = pd.qcut(error_model[f"mse_{i}"], q=2, labels=False).fillna(2).astype(int)
    error_mapping = error_mapping.merge(error_model[["county", f"{model}_base", f"{model}_prop"]], on="county", how="left")
error_mapping.to_csv("../results/book/5/error_mapping.csv", index=False)

error_summary = []
for model in model_iter:
    file_list = [(f"kf{i}", f"_kf{i}") for i in range(1, 6)] + [("mean", "")]
    for index, suffix in file_list:
        error_accu = pd.read_csv(f"../results/work/1/test_accu_{model}{suffix}.csv", engine="python")
        error_accu.columns = ["index", "r2_base", "mse_base", "mae_base", "r2_addi", "mse_addi", "mae_addi"]
        base, prop = error_accu.loc[error_accu["index"] == "pret"].iloc[0], error_accu.loc[error_accu["index"] == "fine"].iloc[0]
        error_inst = pd.read_csv(f"../results/work/1/test_inst_{model}{suffix}.csv", engine="python")
        error_inst = error_inst.groupby("county", as_index=False).mean()
        error_inst = error_inst.merge(county_bin, on="county", how="left", validate="one_to_one")
        mse_group = error_inst.groupby("bin", observed=False)[["mse_base_pret", "mse_addi_fine"]].mean().reindex([0, 1, 2])
        mse_group["mse_diff"] = mse_group["mse_addi_fine"] - mse_group["mse_base_pret"]
        error_summary.append(
            {
                "model": model,
                "index": index,
                "r2_prop": prop["r2_addi"],
                "r2_diff": prop["r2_addi"] - base["r2_base"],
                "mse_prop": prop["mse_addi"],
                "mse_diff": prop["mse_addi"] - base["mse_base"],
                "mae_prop": prop["mae_addi"],
                "mae_diff": prop["mae_addi"] - base["mae_base"],
                "low_mse": mse_group.loc[0, "mse_addi_fine"],
                "low_diff": mse_group.loc[0, "mse_diff"],
                "moderate_mse": mse_group.loc[1, "mse_addi_fine"],
                "moderate_diff": mse_group.loc[1, "mse_diff"],
                "high_mse": mse_group.loc[2, "mse_addi_fine"],
                "high_diff": mse_group.loc[2, "mse_diff"],
            }
        )
error_summary = pd.DataFrame(error_summary)
error_summary.to_csv("../results/book/s5/error_summary1.csv", index=False)

error_summary = []
for p in province_iter:
    for model in model_iter:
        error_accu = pd.read_csv(f"../results/work/3/{p}_test_accu_{model}.csv", engine="python")
        error_accu.columns = ["index", "r2_base", "mse_base", "mae_base", "r2_addi", "mse_addi", "mae_addi"]
        base, prop = error_accu.loc[error_accu["index"] == "pret"].iloc[0], error_accu.loc[error_accu["index"] == "fine"].iloc[0]
        error_inst = pd.read_csv(f"../results/work/3/{p}_test_inst_{model}.csv", engine="python")
        error_inst = error_inst.groupby("county", as_index=False).mean()
        error_inst = error_inst.merge(county_bin_province[p], on="county", how="left", validate="one_to_one")
        mse_group = error_inst.groupby("bin", observed=False)[["mse_base_pret", "mse_addi_fine"]].mean().reindex([0, 1, 2])
        mse_group["mse_diff"] = mse_group["mse_addi_fine"] - mse_group["mse_base_pret"]
        error_summary.append(
            {
                "province": p,
                "model": model,
                "r2_prop": prop["r2_addi"],
                "r2_diff": prop["r2_addi"] - base["r2_base"],
                "mse_prop": prop["mse_addi"],
                "mse_diff": prop["mse_addi"] - base["mse_base"],
                "mae_prop": prop["mae_addi"],
                "mae_diff": prop["mae_addi"] - base["mae_base"],
                "low_mse": mse_group.loc[0, "mse_addi_fine"],
                "low_diff": mse_group.loc[0, "mse_diff"],
                "moderate_mse": mse_group.loc[1, "mse_addi_fine"],
                "moderate_diff": mse_group.loc[1, "mse_diff"],
                "high_mse": mse_group.loc[2, "mse_addi_fine"],
                "high_diff": mse_group.loc[2, "mse_diff"],
            }
        )
error_summary = pd.DataFrame(error_summary).set_index(["province", "model"])
error_summary.to_csv("../results/book/s5/error_summary2.csv")

In [ ]:
### Prepare datasets for robustness analyses across the two subperiods
feat_imptnce = None
for p in ["early", "later"]:
    exec(f"yield_csv = pd.DataFrame(np.concatenate([county_index_{p}[:, 0].reshape(-1, 1), yield_data_{p}.numpy()], 1), columns=['county', 'yield'])")
    yield_grouped = yield_csv.groupby("county")["yield"].agg(["mean", "std"]).reset_index()
    yield_grouped["cv"], yield_grouped["bin"] = yield_grouped["std"] / yield_grouped["mean"], pd.cut(yield_grouped["mean"], bins=3, labels=False)
    yield_edges = pd.cut(yield_grouped["mean"], bins=3, labels=False, retbins=True)[1]
    yield_t1, yield_t2 = yield_edges[1], yield_edges[2]
    print(f"yield_t1, yield_t2 = {yield_t1}, {yield_t2}")
    yield_grouped = yield_grouped.sort_values(["bin", "county"]).reset_index(drop=True)[["county", "bin", "mean", "std", "cv"]]
    yield_grouped.to_csv(f"../results/book/6/yield_grouped_{p}.csv", index=False)

    yield_mapping = yield_grouped[["county", "mean"]].sort_values("county").reset_index(drop=True)
    exec(f"county_{p} = yield_mapping['county'].copy()")
    yield_mapping.to_csv(f"../results/book/6/yield_mapping_{p}.csv", index=False)

    feat = []
    for i in ["obsvd", "resid"]:
        feat_mean = pd.read_csv(f"../results/work/6/{p}_{i}_dnn.csv", engine="python").abs().values[:, 2:].mean(0)
        feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9, 8).mean(0)])
        feat.append(feat_mean / feat_mean.sum())
    feat_p = pd.DataFrame(np.array(feat).T, index=social_columns + natural_columns, columns=[f"{p}_obsvd", f"{p}_resid"])
    feat_p = feat_p.reindex(natural_columns + social_columns[::-1]).rename_axis("feat").reset_index()
    feat_imptnce = feat_p if feat_imptnce is None else feat_imptnce.merge(feat_p, on="feat", how="outer")
feat_imptnce = feat_imptnce.set_index("feat").reindex(natural_columns + social_columns[::-1]).reset_index()
feat_imptnce.to_csv("../results/book/6/feat_imptnce.csv", index=False)

bins = range(0, 9001, 500)
bin_centers, error_group_train = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)], None
for p in ["early", "later"]:
    error_grouped_gather = []
    for i in range(5):
        error_csv = pd.read_csv(f"../results/work/5/{p}_train_inst_dnn_kf{i+1}.csv", engine="python")
        error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
        error_grouped_gather.append([error_grouped.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, 0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped.columns)
    count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
    error = error_grouped_gather.xs("mean", level=1, axis=1)[["mse_addi_pret", "scores_addi_pret", "mse_addi_fine"]]
    error_grouped_p = pd.concat([error_grouped_gather[["bin"]], count, error], axis=1)
    error_grouped_p.columns = ["bin", f"{p}_count", f"{p}_mse_pret", f"{p}_scores", f"{p}_mse_fine"]
    error_group_train = error_grouped_p if error_group_train is None else error_group_train.merge(error_grouped_p, on="bin", how="outer")
error_group_train = error_group_train.sort_values("bin").reset_index(drop=True)
error_group_train.to_csv("../results/book/6/error_group_train.csv", index=False)

for p in ["early", "later"]:
    error_csv = pd.read_csv(f"../results/work/5/{p}_test_inst_dnn.csv", engine="python")
    error_grouped = error_csv.groupby("county", as_index=False).mean()
    error_grouped = error_grouped.merge(county_bin_period[p], on="county", how="left", validate="one_to_one")
    error_grouped = error_grouped[["county", "bin", "mse_base_pret", "mse_addi_fine"]]
    error_grouped.columns = ["county", "bin", "mse_base", "mse_prop"]
    error_grouped = error_grouped.sort_values(["bin", "county"]).reset_index(drop=True)
    error_grouped.to_csv(f"../results/book/6/error_group_{p}.csv", index=False)

error_dataset = []
for p in ["early", "later"]:
    error_csv = pd.read_csv(f"../results/work/5/{p}_test_accu_dnn.csv", engine="python")
    base, prop = error_csv.loc[error_csv["Unnamed: 0"] == "pret"].iloc[0], error_csv.loc[error_csv["Unnamed: 0"] == "fine"].iloc[0]
    error_dataset.append(
        {
            "index": p,
            "r2_base": base["r2_base"],
            "mse_base": base["mse_base"],
            "mae_base": base["mae_base"],
            "r2_prop": prop["r2_addi"],
            "mse_prop": prop["mse_addi"],
            "mae_prop": prop["mae_addi"],
        }
    )
error_dataset = pd.DataFrame(error_dataset)
error_dataset.to_csv("../results/book/6/error_dataset.csv", index=False)

for p in ["early", "later"]:
    error_csv = pd.read_csv(f"../results/work/5/{p}_test_inst_dnn.csv", engine="python")
    error_grouped = error_csv.groupby("county").agg("mean").reset_index()
    error_grouped = error_grouped.sort_values(by="county")[["county", "mse_base_pret", "mse_addi_fine"]]
    exec(f"error_grouped = pd.DataFrame(county_{p}).merge(error_grouped, on='county', how='left')")
    error_grouped.columns = ["county", "mse_base", "mse_prop"]
    for i in ["base", "prop"]:
        exec(f"error_grouped['bin_{i}'] = pd.qcut(error_grouped['mse_{i}'], q=2, labels=False)")
        exec(f"error_grouped['bin_{i}'] = error_grouped['bin_{i}'].fillna(2)")
    error_grouped = error_grouped[["county", "bin_base", "bin_prop"]]
    error_grouped.to_csv(f"../results/book/6/error_mapping_{p}.csv", index=False)

bins = range(0, 9001, 500)
bin_centers = [(bins[i] + bins[i + 1]) // 2 for i in range(len(bins) - 1)]
for p in ["early", "later"]:
    feat_imptnce, error_group_train, error_group_test, error_mapping, error_dataset = None, None, None, None, []
    for model in model_iter[1:]:
        feat = []
        for i in ["obsvd", "resid"]:
            feat_mean = pd.read_csv(f"../results/work/6/{p}_{i}_{model}.csv", engine="python").abs().values[:, 2:].mean(0)
            feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9, 8).mean(0)])
            feat.append(feat_mean / feat_mean.sum())
        feat_model = pd.DataFrame(np.array(feat).T, index=social_columns + natural_columns, columns=[f"{model}_yo", f"{model}_dy"])
        feat_model = feat_model.reindex(natural_columns + social_columns[::-1]).rename_axis("feat").reset_index()
        feat_imptnce = feat_model if feat_imptnce is None else feat_imptnce.merge(feat_model, on="feat", how="outer")

        error_grouped_gather = []
        for i in range(5):
            error_csv = pd.read_csv(f"../results/work/5/{p}_train_inst_{model}_kf{i+1}.csv", engine="python")
            error_csv["bin"] = pd.cut(error_csv["yield"], bins=bins, labels=bin_centers)
            error_grouped = error_csv.groupby("bin", observed=False).agg(["count", "mean"]).reset_index()
            error_grouped_gather.append([error_grouped.values])
        error_grouped_gather = np.concatenate(error_grouped_gather, 0)
        error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
        error_grouped_gather = pd.DataFrame(error_grouped_gather, columns=error_grouped.columns)
        count = error_grouped_gather.xs("count", level=1, axis=1)[["yield"]]
        error = error_grouped_gather.xs("mean", level=1, axis=1)[["mse_addi_pret", "scores_addi_pret", "mse_addi_fine"]]
        train_model = pd.concat([error_grouped_gather[["bin"]], count, error], axis=1)
        train_model.columns = ["bin", "count", f"{model}_mse_pret", f"{model}_scores", f"{model}_mse_fine"]
        if error_group_train is None:
            error_group_train = train_model
        else:
            error_group_train = error_group_train.merge(train_model.drop(columns="count"), on="bin", how="outer")

        error_csv = pd.read_csv(f"../results/work/5/{p}_test_inst_{model}.csv", engine="python")
        error_county = error_csv.groupby("county", as_index=False).mean(numeric_only=True)
        test_model = error_county[["county", "mse_base_pret", "mse_addi_fine"]].merge(county_bin_period[p], on="county", how="left", validate="one_to_one")
        test_model = test_model[["county", "bin", "mse_base_pret", "mse_addi_fine"]]
        test_model.columns = ["county", "bin", f"{model}_mse_base", f"{model}_mse_prop"]
        if error_group_test is None:
            error_group_test = test_model
        else:
            error_group_test = error_group_test.merge(test_model.drop(columns="bin"), on="county", how="outer")

        error_accu = pd.read_csv(f"../results/work/5/{p}_test_accu_{model}.csv", engine="python")
        base = error_accu.loc[error_accu["Unnamed: 0"] == "pret"].iloc[0]
        prop = error_accu.loc[error_accu["Unnamed: 0"] == "fine"].iloc[0]
        error_dataset.append(
            {
                "model": model,
                "r2_base": base["r2_base"],
                "mse_base": base["mse_base"],
                "mae_base": base["mae_base"],
                "r2_prop": prop["r2_addi"],
                "mse_prop": prop["mse_addi"],
                "mae_prop": prop["mae_addi"],
            }
        )

        mapping_model = error_county[["county", "mse_base_pret", "mse_addi_fine"]].copy()
        mapping_model.columns = ["county", "mse_base", "mse_prop"]
        exec(f"mapping_model = pd.DataFrame(county_{p}).merge(mapping_model, on='county', how='left')")
        for i in ["base", "prop"]:
            mapping_model[f"{model}_{i}"] = pd.qcut(mapping_model[f"mse_{i}"], q=2, labels=False).fillna(2).astype(int)
        mapping_model = mapping_model[["county", f"{model}_base", f"{model}_prop"]]
        error_mapping = mapping_model if error_mapping is None else error_mapping.merge(mapping_model, on="county", how="outer")
    feat_imptnce = feat_imptnce.set_index("feat").reindex(natural_columns + social_columns[::-1]).reset_index()
    feat_imptnce.to_csv(f"../results/book/s6/feat_imptnce_{p}.csv", index=False)
    error_group_train.sort_values("bin").reset_index(drop=True).to_csv(f"../results/book/s6/error_group_train_{p}.csv", index=False)
    error_group_test.sort_values(["bin", "county"]).reset_index(drop=True).to_csv(f"../results/book/s6/error_group_test_{p}.csv", index=False)
    pd.DataFrame(error_dataset).to_csv(f"../results/book/s6/error_dataset_{p}.csv", index=False)
    error_mapping.sort_values("county").reset_index(drop=True).to_csv(f"../results/book/s6/error_mapping_{p}.csv", index=False)

yield_t1, yield_t2 = 2927.270653134301, 5297.153118315197
yield_t1, yield_t2 = 3511.646624981916, 5857.545626944083
